# These Functions convert the Dataset into a YOLO Format

## Imports

In [1]:
%load_ext autoreload
%autoreload 2
#automatically reload any imported modules when you re-run a cell

In [2]:
from pathlib import Path

DATA_BASE_DIR = Path("/home/stella/computer_vision/PMOF")


def list_record_ids(data_base_dir):
    data_base_dir = Path(data_base_dir)
    img_dir = data_base_dir / "images"
    if not img_dir.is_dir():
        raise FileNotFoundError(f"Image directory not found: {img_dir}")

    record_ids = [path.name for path in img_dir.iterdir() if path.is_dir() and path.name.startswith("rec")]
    if not record_ids:
        raise FileNotFoundError(f"No record ids found in dataset: {data_base_dir}")

    return sorted(record_ids, key=lambda record_id: int(record_id.replace("rec", "")))

In [3]:
import json
import logging
import time
from collections import Counter

import numpy as np
import yaml

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
logger = logging.getLogger("pmof_dataprep")


def xywhr_to_xyxyxyxy(bbox, rotation):
    tlx, tly, width, height = bbox
    center_x = tlx + width / 2
    center_y = tly + height / 2

    corners = np.array([
        [-width / 2, -height / 2],
        [width / 2, -height / 2],
        [width / 2, height / 2],
        [-width / 2, height / 2],
    ])

    theta = np.radians(rotation)
    rotation_matrix = np.array([
        [np.cos(theta), -np.sin(theta)],
        [np.sin(theta), np.cos(theta)],
    ])

    return (corners @ rotation_matrix.T + np.array([center_x, center_y])).flatten().tolist()

In [4]:
PERSON_YOLO_NAMES = {0: "person"}
ACTION_YOLO_NAMES = {
    0: "standing",
    1: "seated",
    2: "lying",
    3: "seated_ground",
    #4: "asking_for_help",
}

ACTION_ALIASES = {
    "standing": "standing",
    "walking": "standing",
    "seated": "seated",
    "lying": "lying",
    "falling": "lying",
    "on the ground": "lying",
    "on_ground": "lying",
    "seated on the ground": "seated_ground",
    "seated_ground": "seated_ground",
    "seated_on_the_ground": "seated_ground",
    "asking for help": "asking_for_help",
    "asking_for_help": "asking_for_help",
    "aksing for help": "asking_for_help",
}


def normalize_action_label(action, unknown_action_policy="error"):
    """Normalize PMOF action labels into the YOLO action taxonomy."""
    if action is None:
        if unknown_action_policy == "skip":
            return None
        raise ValueError("Missing action attribute for person annotation.")

    normalized = str(action).strip().lower().replace("-", "_")
    normalized = "_".join(normalized.split())
    normalized = ACTION_ALIASES.get(normalized, ACTION_ALIASES.get(normalized.replace("_", " ")))

    if normalized is None:
        if unknown_action_policy == "skip":
            return None
        known_actions = ", ".join(sorted(ACTION_ALIASES))
        raise ValueError(f"Unknown action label {action!r}. Known labels: {known_actions}")

    return normalized


def coco_to_yolo(
    data_base_dir,
    record_id,
    action_labels=False,
    include_occluded=False,
    unknown_action_policy="error",
    min_box_size=None,
):
    """
    Convert COCO-format annotations to YOLO OBB txt files.

    Object classes are ignored. Use action_labels=False for one person class, or
    action_labels=True for normalized action classes.
    """
    data_base_dir = Path(data_base_dir)
    yolo_names = ACTION_YOLO_NAMES if action_labels else PERSON_YOLO_NAMES
    yolo_name_to_id = {name: class_id for class_id, name in yolo_names.items()}
    action_counts = Counter()
    print(f"Using YOLO names: {yolo_names}")

    output_dir = data_base_dir / "labels" / record_id
    output_dir.mkdir(parents=True, exist_ok=True)

    annotation_path = data_base_dir / "annotations" / f"{record_id}_annotations.json"
    with open(annotation_path, "r") as f:
        all_annotations = json.load(f)

    images = {str(img["id"]): img for img in all_annotations["images"]}
    category_id_name_match = {cat["id"]: cat["name"] for cat in all_annotations["categories"]}

    anns_by_image = {}
    for ann in all_annotations["annotations"]:
        img_id = str(ann["image_id"])
        anns_by_image.setdefault(img_id, []).append(ann)

    for image_id, image_data in images.items():
        #start = time.time()
        image_width = image_data["width"]
        image_height = image_data["height"]
        file_stem = Path(image_data["file_name"]).stem

        yolo_annotations = []
        for ann in anns_by_image.get(image_id, []):
            cls_name = category_id_name_match.get(ann["category_id"])
            if cls_name != "person":
                continue

            attributes = ann.get("attributes", {})
            if attributes.get("occluded", False) and not include_occluded:
                continue

            if action_labels:
                cls_name = normalize_action_label(attributes.get("action"), unknown_action_policy)
                if cls_name is None:
                    continue
            else:
                cls_name = "person"

            yolo_cat_id = yolo_name_to_id[cls_name]
            bbox = ann["bbox"]
            if min_box_size is not None and (bbox[2] < min_box_size or bbox[3] < min_box_size):
                continue

            rotation = attributes.get("rotation", 0)
            xyxyxyxy = xywhr_to_xyxyxyxy(bbox, rotation)
            xyxyxyxy_n = [
                xyxyxyxy[0] / image_width,
                xyxyxyxy[1] / image_height,
                xyxyxyxy[2] / image_width,
                xyxyxyxy[3] / image_height,
                xyxyxyxy[4] / image_width,
                xyxyxyxy[5] / image_height,
                xyxyxyxy[6] / image_width,
                xyxyxyxy[7] / image_height,
            ]

            annotation_line = f"{yolo_cat_id} " + " ".join(f"{coord:.6f}" for coord in xyxyxyxy_n)
            yolo_annotations.append(annotation_line)
            if action_labels:
                action_counts[cls_name] += 1

        with open(output_dir / f"{file_stem}.txt", "w") as txt_file:
            txt_file.write("\n".join(yolo_annotations))

        #end = time.time()
        #print(f"Processing image {image_id} took {end - start:.4f} sec")

    return action_counts

In [5]:
record_ids = list_record_ids(DATA_BASE_DIR)
record_ids

['rec0',
 'rec1',
 'rec2',
 'rec3',
 'rec4',
 'rec5',
 'rec6',
 'rec7',
 'rec8',
 'rec9',
 'rec10',
 'rec11',
 'rec12',
 'rec13',
 'rec14',
 'rec15',
 'rec16',
 'rec17',
 'rec18',
 'rec19',
 'rec20',
 'rec21',
 'rec22',
 'rec23',
 'rec24',
 'rec25',
 'rec26',
 'rec27',
 'rec28',
 'rec29',
 'rec30',
 'rec31',
 'rec32',
 'rec39',
 'rec41',
 'rec42',
 'rec45',
 'rec52',
 'rec59',
 'rec60',
 'rec61']

In [7]:
#val_record_ids = ["rec26", "rec27", "rec28", "rec29", "rec30"]
#test_record_ids = ["rec31", "rec32", "rec42"]
#train_record_ids = [record_id for record_id in record_ids if record_id not in val_record_ids and record_id not in test_record_ids and record_id != "rec0"]

#modell for enableATO testday september 2026
#train_record_ids = ['rec4', "rec22", "rec25", "rec27", "rec28", "rec29", "rec30", 
#                    "rec52", "rec59", "rec60", "rec61",
#                    "rec31", "rec39", "rec41", "rec42", "rec45"]
#val_record_ids = ["rec32"]
#test_record_ids = ["rec31", "rec42"]

#model for enableATO symposium
train_record_ids = ['rec4', "rec22", "rec25", "rec29", "rec30"]
val_record_ids = ['rec27']
test_record_ids = ['rec28']

print(f"Train records ({len(train_record_ids)}): {train_record_ids}")
print(f"Val records ({len(val_record_ids)}): {val_record_ids}")
print(f"Test records ({len(test_record_ids)}): {test_record_ids}")

Train records (5): ['rec4', 'rec22', 'rec25', 'rec29', 'rec30']
Val records (1): ['rec27']
Test records (1): ['rec28']


In [8]:
LABEL_MODE = "action"  # "person" or "action"
if LABEL_MODE not in {"person", "action"}:
    raise ValueError("LABEL_MODE must be 'person' or 'action'.")

use_action_labels = LABEL_MODE == "action"
split_record_ids = {"train": train_record_ids, "val": val_record_ids, "test": test_record_ids}
split_action_counts = {}
for split_name, split_ids in split_record_ids.items():
    split_counts = Counter()
    for record_id in split_ids:
        print(f"Starting with {split_name} record id: {record_id}")
        split_counts.update(coco_to_yolo(DATA_BASE_DIR, record_id, action_labels=use_action_labels)) #min_box_size=25 or min_box_size=50 for ATO Testday Sep. 2026

    split_action_counts[split_name] = {
        action: split_counts[action] for action in ACTION_YOLO_NAMES.values()
    }
    print(f"{split_name.title()} action distribution: {split_action_counts[split_name]}")

Starting with train record id: rec4
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Starting with train record id: rec22
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Starting with train record id: rec25
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Starting with train record id: rec29
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Starting with train record id: rec30
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Train action distribution: {'standing': 959, 'seated': 5282, 'lying': 199, 'seated_ground': 55}
Starting with val record id: rec27
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Val action distribution: {'standing': 76, 'seated': 747, 'lying': 83, 'seated_ground': 39}
Starting with test record id: rec28
Using YOLO names: {0: 'standing', 1: 'seated', 2: 'lying', 3: 'seated_ground'}
Test 

# Generate train.txt & val.txt Files

In [9]:
def generate_data_txt_for_sequences(base_dirs_with_sequences, output_file="val.txt", max_entries=None):
    """
    Generate a YOLO image-list txt for selected sequences.

    Missing label files are created as empty files so background images remain valid.
    """
    valid_entries = []

    for base_dir, sequences in base_dirs_with_sequences:
        base_dir = Path(base_dir)
        labels_root = base_dir / "labels"
        images_root = base_dir / "images"

        for seq in sequences:
            labels_seq_dir = labels_root / seq
            images_seq_dir = images_root / seq

            if not labels_seq_dir.is_dir() or not images_seq_dir.is_dir():
                raise FileNotFoundError(f"Missing image or label directory for {seq} in {base_dir}")
            logger.info(f"Found {labels_seq_dir} and {images_seq_dir}")

            for image_path in sorted(images_seq_dir.iterdir()):
                if image_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                    continue

                label_path = labels_seq_dir / f"{image_path.stem}.txt"
                label_path.touch(exist_ok=True)
                valid_entries.append(str(image_path.resolve()))

                if max_entries is not None and len(valid_entries) >= max_entries:
                    break

            if max_entries is not None and len(valid_entries) >= max_entries:
                break

        if max_entries is not None and len(valid_entries) >= max_entries:
            break

    nr_entries = len(valid_entries)
    if nr_entries == 0:
        raise FileNotFoundError(f"No entries for {output_file}.")

    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    output_file.write_text("\n".join(valid_entries) + "\n")

    print(f"{output_file} created with {nr_entries} entries")
    return nr_entries, output_file

In [11]:
datasets_train = [(DATA_BASE_DIR, train_record_ids)]
datasets_val = [(DATA_BASE_DIR, val_record_ids)]
datasets_test = [(DATA_BASE_DIR, test_record_ids)]

generate_data_txt_for_sequences(datasets_train, output_file=DATA_BASE_DIR / "train.txt")
generate_data_txt_for_sequences(datasets_val, output_file=DATA_BASE_DIR / "val.txt")
generate_data_txt_for_sequences(datasets_test, output_file=DATA_BASE_DIR / "test.txt")

2026-09-11 19:57:09,885 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec4 and /home/stella/computer_vision/PMOF/images/rec4
2026-09-11 19:57:09,911 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec22 and /home/stella/computer_vision/PMOF/images/rec22
2026-09-11 19:57:09,935 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec25 and /home/stella/computer_vision/PMOF/images/rec25
2026-09-11 19:57:09,956 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec29 and /home/stella/computer_vision/PMOF/images/rec29
2026-09-11 19:57:09,972 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec30 and /home/stella/computer_vision/PMOF/images/rec30
2026-09-11 19:57:09,985 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec27 and /home/stella/computer_vision/PMOF/images/rec27
2026-09-11 19:57:09,995 | INFO | Found /home/stella/computer_vision/PMOF/labels/rec28 and /home/stella/computer_vision/PMOF/images/rec28


/home/stella/computer_vision/PMOF/train.txt created with 3649 entries
/home/stella/computer_vision/PMOF/val.txt created with 477 entries
/home/stella/computer_vision/PMOF/test.txt created with 584 entries


(584, PosixPath('/home/stella/computer_vision/PMOF/test.txt'))

# Write yaml-file

In [12]:
yaml_text = dict(
    path=str(DATA_BASE_DIR),
    train="train.txt",
    val="val.txt",
    test="test.txt",
    names=ACTION_YOLO_NAMES if LABEL_MODE == "action" else PERSON_YOLO_NAMES,
)

with open(DATA_BASE_DIR / f"PMOF_{LABEL_MODE}_symposium.yaml", "w") as outfile:
    yaml.dump(yaml_text, outfile, default_flow_style=False, sort_keys=False)